# NextAI — Train the compact engineering student

This notebook fine-tunes one Qwen3.5-0.8B student for CAD, BIM and IFC. It uses the committed Next3D corpus-preparation and training scripts. The larger CAD/BIM teacher models are not loaded into the final app or copied into the installer. The training corpus, image assets, adapter and checkpoints remain in this Colab/Google Drive workspace.

**Use a GPU runtime.** The first pass uses BF16/FP16 LoRA, not QLoRA. It begins with 1,000 Autodesk CAD rows and a bounded BIM sample. Increase the sample only after the first run completes successfully.

The test split remains held out. Evaluation metrics are diagnostic only; native NextCAD, Next BIM and IFC validation are still required before accepting the student.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU (or another CUDA GPU), then rerun this cell.')
props = torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(props.total_memory / (1024**3), 2))
print('BF16 supported:', torch.cuda.is_bf16_supported())


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
WORKSPACE = Path('/content/nextai-workspace')
WORKSPACE.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT = Path('/content/drive/MyDrive/NextAI-training/first-run')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Training outputs:', DRIVE_OUTPUT)


In [ ]:
import os
repo = WORKSPACE / 'Next3D'
if not repo.exists():
    os.system('git clone --depth 1 --branch nextai-operator-foundation https://github.com/28royalpapi/Next3D.git ' + str(repo))
assert (repo / 'training/nextai/prepare_public_corpora.py').is_file()
assert (repo / 'training/nextai/train_nextai.py').is_file()
assert (repo / 'training/nextai/evaluate_nextai.py').is_file()
print('Next3D training scripts ready:', repo)


In [ ]:
%pip -q install --upgrade datasets pillow


In [ ]:
corpus = WORKSPACE / 'corpus-first-run'
cmd = [
    'python', str(repo / 'training/nextai/prepare_public_corpora.py'),
    '--output-dir', str(corpus),
    '--max-cad-train', '1000',
    '--max-cad-validation', '150',
    '--max-cad-test', '150',
    '--max-bim-train', '10000',
    '--max-bim-validation', '1500',
    '--max-ifcqa-train', '2000',
    '--max-ifcqa-test', '400',
    '--max-mixed-train', '8000',
    '--max-validation', '1000',
    '--max-test', '500',
]
import subprocess
subprocess.run(cmd, check=True)


In [ ]:
import json
manifest = json.loads((corpus / 'manifest.json').read_text())
print(json.dumps({
    'train_records': manifest['train_records'],
    'validation_records': manifest['validation_records'],
    'test_records': manifest['test_records'],
    'training_domain_counts': manifest['training_domain_counts'],
    'source_counts': manifest['source_counts'],
}, indent=2))
assert manifest['train_records'] > 0 and manifest['validation_records'] > 0 and manifest['test_records'] > 0
assert manifest['test_not_used_for_training'] is True


## Install the training framework

This environment is separate from the Next3D desktop package. It is not added to the 1.2 GB installer.

In [ ]:
%pip -q install --upgrade unsloth unsloth_zoo transformers trl peft accelerate bitsandbytes pillow


In [ ]:
run_dir = WORKSPACE / 'student-run-001'
train_cmd = [
    'python', str(repo / 'training/nextai/train_nextai.py'),
    '--data-dir', str(corpus),
    '--output-dir', str(run_dir),
    '--base-model', 'Qwen/Qwen3.5-0.8B',
    '--max-steps', '800',
    '--max-seq-length', '2048',
    '--batch-size', '1',
    '--gradient-accumulation', '8',
    '--learning-rate', '0.0001',
    '--lora-rank', '16',
]
subprocess.run(train_cmd, check=True)
print('Student training finished:', run_dir)


In [ ]:
base_report = WORKSPACE / 'eval-base.json'
tuned_report = WORKSPACE / 'eval-tuned.json'
test_file = corpus / 'test.jsonl'
subprocess.run([
    'python', str(repo / 'training/nextai/evaluate_nextai.py'),
    '--model', 'Qwen/Qwen3.5-0.8B',
    '--test-jsonl', str(test_file),
    '--output', str(base_report),
], check=True)
subprocess.run([
    'python', str(repo / 'training/nextai/evaluate_nextai.py'),
    '--model', str(run_dir / 'adapter'),
    '--test-jsonl', str(test_file),
    '--output', str(tuned_report),
], check=True)
print('Base and tuned student reports saved.')


In [ ]:
import shutil
artifacts = [run_dir / 'adapter', run_dir / 'gguf_q4_k_m', run_dir / 'training_result.json', base_report, tuned_report, corpus / 'manifest.json']
for path in artifacts:
    if not path.exists():
        continue
    destination = DRIVE_OUTPUT / path.name
    if path.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(path, destination)
    else:
        shutil.copy2(path, destination)
print('Saved student adapter, GGUF, measured model/projector checksums and evaluation reports to:', DRIVE_OUTPUT)
print('Training checkpoints and raw training corpus remain in the temporary Colab workspace; they are not part of the product package.')
print('The final model is only a candidate until native CAD/BIM validation and full package-size measurements pass.')
